In [ ]:
pip install transformers datasets peft accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [ ]:
pip install --upgrade transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 52.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2


In [ ]:
!pip install evaluate
!pip install --upgrade datasets
!pip install --upgrade pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 18.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 45.1 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas=

In [ ]:
import os
import math
import time
import json
from dataclasses import dataclass
from typing import Dict, List, Callable, Tuple

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, get_peft_model
import evaluate

In [ ]:
# GLOBAL CONFIGURATION & DATA LOADING

# Global configuration
BASE_MODEL_NAME = "distilbert-base-uncased"
DATASET_NAME = "dair-ai/emotion"
MAX_LENGTH = 128
TRAIN_SUBSET_SIZE = 3000
NUM_EPOCHS = 3  # Each trial trains for exactly 3 epochs
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")

# Load accuracy metric
accuracy_metric = evaluate.load("accuracy")


def load_and_prepare_data():

    print("Loading dataset...")
    raw_datasets = load_dataset(DATASET_NAME)

    # Use 3,000 examples from train split
    train_dataset = raw_datasets["train"].shuffle(seed=42).select(range(TRAIN_SUBSET_SIZE))
    val_dataset = raw_datasets["validation"]
    test_dataset = raw_datasets["test"]

    print(f"Train subset size: {len(train_dataset)}")
    print(f"Validation size: {len(val_dataset)}")
    print(f"Test size: {len(test_dataset)}")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

    def preprocess_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )

    print("Tokenizing datasets...")
    tokenized_train = train_dataset.map(preprocess_function, batched=True)
    tokenized_val = val_dataset.map(preprocess_function, batched=True)
    tokenized_test = test_dataset.map(preprocess_function, batched=True)

    # Rename label column
    tokenized_train = tokenized_train.rename_column("label", "labels")
    tokenized_val = tokenized_val.rename_column("label", "labels")
    tokenized_test = tokenized_test.rename_column("label", "labels")

    # Set format
    tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    tokenized_val.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    # Get label info
    num_labels = raw_datasets["train"].features["label"].num_classes
    id2label = {i: l for i, l in enumerate(raw_datasets["train"].features["label"].names)}
    label2id = {v: k for k, v in id2label.items()}

    print("Dataset preparation complete!\n")

    return {
        "tokenized_train": tokenized_train,
        "tokenized_val": tokenized_val,
        "tokenized_test": tokenized_test,
        "tokenizer": tokenizer,
        "num_labels": num_labels,
        "id2label": id2label,
        "label2id": label2id
    }

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# HYPERPARAMETER SPACE DEFINITION

def lora_hyperparameter_space():

    search_space = {
        "learning_rate": {"type": "log_uniform", "min": 5e-6, "max": 5e-4},
        "warmup_ratio":  {"type": "continuous",   "min": 0.0,  "max": 0.2},
        "lora_r":        {"type": "discrete",     "values": [2, 4, 8, 16, 32]},
        "lora_alpha":    {"type": "discrete",     "values": [8, 16, 32, 64, 128]},
        "lora_dropout":  {"type": "continuous",   "min": 0.0,  "max": 0.3},
        "target_modules": {"type": "categorical", "values": [
            "attention-only",
            "attention-plus-ffn"
        ]}
    }


    def decode(x):
        # Learning rate (log-uniform)
        lr_min = search_space["learning_rate"]["min"]
        lr_max = search_space["learning_rate"]["max"]
        log_min = math.log(lr_min)
        log_max = math.log(lr_max)
        log_lr = log_min + x[0] * (log_max - log_min)
        learning_rate = math.exp(log_lr)

        # Warmup ratio (continuous)
        warmup_min = search_space["warmup_ratio"]["min"]
        warmup_max = search_space["warmup_ratio"]["max"]
        warmup_ratio = warmup_min + x[1] * (warmup_max - warmup_min)

        # Discrete parameters
        def pick_discrete(val, choices):
            idx = int(round(val * (len(choices) - 1)))
            return choices[max(0, min(len(choices)-1, idx))]

        lora_r = pick_discrete(x[2], search_space["lora_r"]["values"])
        lora_alpha = pick_discrete(x[3], search_space["lora_alpha"]["values"])

        # LoRA dropout (continuous)
        dropout_min = search_space["lora_dropout"]["min"]
        dropout_max = search_space["lora_dropout"]["max"]
        lora_dropout = dropout_min + x[4] * (dropout_max - dropout_min)

        # Target modules (categorical)
        if x[5] < 0.5:
            target_modules = ["q_lin", "v_lin"]  # attention only
        else:
            target_modules = ["q_lin", "v_lin", "ffn.lin1", "ffn.lin2"]  # attention + FFN

        return {
            "learning_rate": learning_rate,
            "warmup_ratio": warmup_ratio,
            "lora_r": lora_r,
            "lora_alpha": lora_alpha,
            "lora_dropout": lora_dropout,
            "target_modules": target_modules,
        }

    return search_space, decode

# helper to map raw module list to label
def map_target_modules_label(modules: List[str]) -> str:

    attention_keys = {"q_lin", "v_lin"}
    ffn_keys = {"ffn.lin1", "ffn.lin2"}
    has_attention = any(m in attention_keys for m in modules)
    has_ffn = any(m in ffn_keys for m in modules)
    if has_attention and has_ffn:
        return "attention-plus-ffn"
    elif has_attention:
        return "attention-only"
    else:
        # fallback if something unexpected happens
        return "attention-only"


In [ ]:
# LORA TRAINING & EVALUATION FUNCTION

def compute_metrics(eval_pred):
    """Compute accuracy for Trainer"""
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)


def train_and_evaluate_lora(
    hyperparams: Dict,
    data_dict: Dict,
    seed: int = 42,
    trial_no: int = None
) -> Tuple[float, float, float]:

    start_time = time.time()

    run_name = f"dpso_ga_trial{trial_no}_seed{seed}" if trial_no else f"trial_seed{seed}"

    set_seed(seed)

    # reset GPU peak memory stats at start of trial
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    # Load base model
    base_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_NAME,
        num_labels=data_dict["num_labels"],
        id2label=data_dict["id2label"],
        label2id=data_dict["label2id"],
    )

    # Configure LoRA
    lora_config = LoraConfig(
        r=hyperparams["lora_r"],
        lora_alpha=hyperparams["lora_alpha"],
        lora_dropout=hyperparams["lora_dropout"],
        target_modules=hyperparams["target_modules"],
        bias="none",
        task_type="SEQ_CLS",
    )

    # Apply LoRA to model
    model = get_peft_model(base_model, lora_config)
    model.to(DEVICE)

    # Training arguments
    training_args = TrainingArguments(
        output_dir=f"./tmp/{run_name}",
        num_train_epochs=NUM_EPOCHS,  # Always 3 epochs
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=hyperparams["learning_rate"],
        warmup_ratio=hyperparams["warmup_ratio"],
        weight_decay=0.01,
        logging_strategy="epoch",
        save_strategy="no",
        report_to="none",
        load_best_model_at_end=False,
        remove_unused_columns=True,
        seed=seed,
    )

    # Create Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=data_dict["tokenized_train"],
        eval_dataset=data_dict["tokenized_val"],
        tokenizer=data_dict["tokenizer"],
        compute_metrics=compute_metrics,
    )

    # Train for 3 epochs
    trainer.train()

    # Evaluate on validation set
    eval_metrics = trainer.evaluate(eval_dataset=data_dict["tokenized_val"])
    val_acc = eval_metrics.get("eval_accuracy", float("nan"))

    # Calculate training time
    train_time = time.time() - start_time

    # read peak GPU memory and convert to GB
    if torch.cuda.is_available():
        peak_bytes = torch.cuda.max_memory_allocated()
        gpu_memory_gb = float(peak_bytes) / (1024 ** 3)
    else:
        gpu_memory_gb = 0.0


    # Clean up
    del trainer
    del model
    del base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return val_acc, train_time, gpu_memory_gb

In [ ]:
# HYBRID DPSO-GA ALGORITHM

@dataclass
class Particle:
    """Represents a particle (candidate solution) in the swarm"""
    position: np.ndarray
    velocity: np.ndarray
    fitness: float
    best_position: np.ndarray
    best_fitness: float
    train_time: float = 0.0


class HybridDPSO_GA:


    def __init__(
        self,
        fitness_function: Callable,
        decode_function: Callable,
        search_space: Dict,
        n_dimensions: int = 6,
        total_trials: int = 10,
        initial_population: int = 5,
        omega: float = 0.1,
        c1: float = 0.5,
        c2: float = 0.5,
        ga_crossover_prob: float = 0.8,
        ga_mutation_prob: float = 0.1,
        seed: int = 42
    ):

        self.fitness_function = fitness_function
        self.decode_function = decode_function
        self.search_space = search_space
        self.n_dimensions = n_dimensions
        self.initial_population = initial_population
        self.population_size = initial_population
        self.omega = omega
        self.c1 = c1
        self.c2 = c2
        self.ga_crossover_prob = ga_crossover_prob
        self.ga_mutation_prob = ga_mutation_prob

        np.random.seed(seed)
        self.seed = seed

        self.max_iterations = total_trials

        # Swarm state
        self.swarm: List[Particle] = []
        self.global_best_position = None
        self.global_best_fitness = -np.inf
        self.global_best_hyperparams = None

        # Trial tracking
        self.trials_evaluated = 0
        self.trial_history = [] #list of per-evaluation logs (candidates)

    def _evaluate_position(
        self,
        position: np.ndarray,
        trial_no: int,
        operation: str,
        particle_idx: int
    ) -> Tuple[float, float, float]:

        hyperparams = self.decode_function(position)

        raw_modules = hyperparams.get("target_modules", [])
        target_label = map_target_modules_label(raw_modules)

        fitness, train_time, gpu_memory_gb = self.fitness_function(hyperparams, trial_no)

        self.trials_evaluated += 1

        # store label form of target_modules in JSON
        hyperparams_for_log = dict(hyperparams)
        hyperparams_for_log["target_modules"] = target_label


        # Record trial / candidate information
        self.trial_history.append({
            "trial_no": trial_no,
            "operation": operation,
            "particle_idx": int(particle_idx),
            "val_acc": float(fitness),
            "train_time_sec": float(train_time),
            "gpu_memory_gb": float(gpu_memory_gb),
            "seed": self.seed + trial_no,
            "hyperparams": hyperparams_for_log,
            "raw_position": position.tolist()
        })

        return fitness, train_time, gpu_memory_gb

    def initialize_swarm(self):
        """
        Initialize swarm with initial_population particles.
        Uses first N trials.
        """
        print(f"Initializing swarm with {self.initial_population} particles...")
        print(f"These are initial evaluations before  {self.max_iterations} DPSO+GA iterations.\n")

        for i in range(self.initial_population):
            trial_no = self.trials_evaluated + 1
            position = np.random.rand(self.n_dimensions)
            velocity = np.zeros(self.n_dimensions)

            print(f"Initial Trial {trial_no}: Evaluating initial particle {i+1}/{self.initial_population}...")
            fitness, train_time, gpu_memory_gb = self._evaluate_position(
                position=position,
                trial_no=trial_no,
                operation="init",
                particle_idx=i
            )

            particle = Particle(
                position=position.copy(),
                velocity=velocity,
                fitness=fitness,
                best_position=position.copy(),
                best_fitness=fitness,
                train_time=train_time
            )

            self.swarm.append(particle)

            # Update global best
            if fitness > self.global_best_fitness:
                self.global_best_fitness = fitness
                self.global_best_position = position.copy()
                self.global_best_hyperparams = self.decode_function(position)

            # Log best_val_acc_so_far for this evaluation
            self.trial_history[-1]["best_val_acc_so_far"] = float(self.global_best_fitness)
            print(f"  Fitness: {fitness:.4f}, Time: {train_time:.1f}s, Best so far: {self.global_best_fitness:.4f}\n")

        print(f"Initialization complete. Initial evaluations: {self.trials_evaluated}\n")

    def _clip_position(self, position: np.ndarray) -> np.ndarray:
        """Clip position to [0,1] bounds"""
        return np.clip(position, 0.0, 1.0)

    def _mutation_operator_F1(self, position: np.ndarray) -> np.ndarray:
        """F1: Mutation operator"""
        new_position = position.copy()
        for i in range(self.n_dimensions):
            if np.random.rand() < self.omega:
                new_position[i] = np.random.rand()
        return self._clip_position(new_position)

    def _crossover_operator_F2(self, tau: np.ndarray, pbest: np.ndarray) -> np.ndarray:
        """F2: Cognitive crossover"""
        delta = tau.copy()
        for i in range(self.n_dimensions):
            if np.random.rand() < self.c1:
                delta[i] = pbest[i]
        return self._clip_position(delta)

    def _crossover_operator_F3(self, delta: np.ndarray, gbest: np.ndarray) -> np.ndarray:
        """F3: Social crossover"""
        new_position = delta.copy()
        for i in range(self.n_dimensions):
            if np.random.rand() < self.c2:
                new_position[i] = gbest[i]
        return self._clip_position(new_position)

    def _dpso_update_particle(self, particle: Particle) -> np.ndarray:

        # F1: Mutation
        tau = self._mutation_operator_F1(particle.position)

        # F2: Cognitive crossover
        delta = self._crossover_operator_F2(tau, particle.best_position)

        # F3: Social crossover
        new_position = self._crossover_operator_F3(delta, self.global_best_position)

        return new_position

    def _ga_crossover(self, parent1_pos: np.ndarray, parent2_pos: np.ndarray) -> np.ndarray:
        """
        GA uniform crossover: create offspring by combining two parents.
        Applied with probability ga_crossover_prob.
        """
        if np.random.rand() >= self.ga_crossover_prob:
            return parent1_pos.copy()

        # Uniform crossover: randomly take each gene from parent1 or parent2
        mask = np.random.rand(self.n_dimensions) < 0.5
        offspring = np.where(mask, parent1_pos, parent2_pos)

        return self._clip_position(offspring)

    def _ga_mutation(self, position: np.ndarray) -> np.ndarray:
        """
        GA mutation: randomly modify genes with probability ga_mutation_prob.
        """
        new_position = position.copy()

        for i in range(self.n_dimensions):
            if np.random.rand() < self.ga_mutation_prob:
                new_position[i] = np.random.rand()

        return self._clip_position(new_position)

    def _tournament_selection(self, k: int = 3) -> Particle:
        """
        Tournament selection: randomly select k particles, return the best one.
        """
        tournament = np.random.choice(self.swarm, size=min(k, len(self.swarm)), replace=False)
        return max(tournament, key=lambda p: p.fitness)

    # optimize runs max_iterations, and in each iteration applies DPSO to ALL particles and then GA to ALL particles.
    def optimize(self, verbose: bool = True) -> Dict:

        print(f"HYBRID DPSO-GA OPTIMIZATION")
        print(f"Total iterations: {self.max_iterations}")
        print()

        # Phase 1: Initialize swarm (evaluate initial population)
        self.initialize_swarm()

        # Phase 2: Run DPSO+GA for max_iterations
        for iteration in range(1, self.max_iterations + 1):
            print(f"{'-'*60}")
            print(f"Iteration {iteration}/{self.max_iterations}")
            print(f"{'-'*60}")

            # DPSO PHASE (update all particles)
            print("DPSO Phase: updating all particles with F1→F2→F3...")
            for i, particle in enumerate(self.swarm):
                # Generate new position via DPSO operators
                new_position = self._dpso_update_particle(particle)

                # Evaluate new position
                trial_no = self.trials_evaluated + 1
                fitness, train_time, gpu_memory_gb = self._evaluate_position(
                    position=new_position,
                    trial_no=trial_no,
                    operation="DPSO",
                    particle_idx=i
                )

                # Update particle
                particle.position = new_position
                particle.fitness = fitness
                particle.train_time = train_time

                # Update personal best
                if fitness > particle.best_fitness:
                    particle.best_fitness = fitness
                    particle.best_position = new_position.copy()

                # Update global best
                if fitness > self.global_best_fitness:
                    self.global_best_fitness = fitness
                    self.global_best_position = new_position.copy()
                    self.global_best_hyperparams = self.decode_function(new_position)

                # Log best_val_acc_so_far
                self.trial_history[-1]["best_val_acc_so_far"] = float(self.global_best_fitness)

            # GA PHASE (apply GA to whole population)
            print("GA Phase: selection + crossover + mutation for all particles...")
            new_swarm: List[Particle] = []
            pop_size = len(self.swarm)

            for j in range(pop_size):
                # Tournament selection: pick two parents from current (DPSO-updated) swarm
                parent1 = self._tournament_selection(k=3)
                parent2 = self._tournament_selection(k=3)

                # Crossover to get offspring
                offspring_pos = self._ga_crossover(parent1.position, parent2.position)

                # Mutation
                offspring_pos = self._ga_mutation(offspring_pos)

                # Evaluate offspring
                trial_no = self.trials_evaluated + 1
                fitness, train_time, gpu_memory_gb = self._evaluate_position(
                    position=offspring_pos,
                    trial_no=trial_no,
                    operation="GA",
                    particle_idx=j
                )

                # Create new particle for next generation
                new_particle = Particle(
                    position=offspring_pos.copy(),
                    velocity=np.zeros(self.n_dimensions),
                    fitness=fitness,
                    best_position=offspring_pos.copy(),
                    best_fitness=fitness,
                    train_time=train_time
                )

                # Update global best
                if fitness > self.global_best_fitness:
                    self.global_best_fitness = fitness
                    self.global_best_position = offspring_pos.copy()
                    self.global_best_hyperparams = self.decode_function(offspring_pos)

                # Log best_val_acc_so_far
                self.trial_history[-1]["best_val_acc_so_far"] = float(self.global_best_fitness)

                new_swarm.append(new_particle)

            # Replace entire swarm with GA-generated offspring
            self.swarm = new_swarm

            # per-iteration summary
            iter_best = max(p.fitness for p in self.swarm)
            iter_mean = float(np.mean([p.fitness for p in self.swarm]))
            print(f"Iteration {iteration} summary: best={iter_best:.4f}, mean={iter_mean:.4f}, global_best={self.global_best_fitness:.4f}\n")

        print("OPTIMIZATION COMPLETE")
        print(f"Seed: {self.seed}")
        print(f"Total objective evaluations: {self.trials_evaluated}")
        print(f"Best validation accuracy: {self.global_best_fitness:.4f}")
        print(f"\nBest hyperparameters:")
        for key, value in self.global_best_hyperparams.items():
            print(f"  {key}: {value}")

        # Aggregate statistics for THIS seed
        all_fitnesses = [t["val_acc"] for t in self.trial_history]
        val_acc_mean = float(np.mean(all_fitnesses))
        val_acc_std = float(np.std(all_fitnesses))
        avg_time = float(np.mean([t["train_time_sec"] for t in self.trial_history]))
        total_time = float(np.sum([t["train_time_sec"] for t in self.trial_history]))


        # compute average GPU memory over trials
        gpu_mems = [t.get("gpu_memory_gb", 0.0) for t in self.trial_history]
        avg_gpu_memory_gb = float(np.mean(gpu_mems)) if gpu_mems else 0.0


        # Find best trial number
        best_trial_no = None
        for t in self.trial_history:
            if abs(t["val_acc"] - self.global_best_fitness) < 1e-6:
                best_trial_no = t["trial_no"]
                break

        # Build trials/candidates list for this seed
        trials_by_no = sorted(self.trial_history, key=lambda x: x["trial_no"])
        trials_for_seed = []
        for entry in trials_by_no:
            trial_no = entry["trial_no"]

            # compute iteration index from trial_no and population size
            pop = self.population_size
            if trial_no <= pop:
                iteration = 0
            else:
                iteration = 1 + (trial_no - pop - 1) // (2 * pop)

            trial_record = {
                "iteration": int(iteration),
                "trial_no": trial_no,
                "best_val_acc_so_far": entry["best_val_acc_so_far"],
                "particles": [
                    {
                        "particle_no": trial_no,  # one candidate per evaluation
                        "operation": entry["operation"],          # init / DPSO / GA
                        "particle_idx": entry["particle_idx"],
                        "val_acc": entry["val_acc"],
                        "train_time_sec": entry["train_time_sec"],
                        "gpu_memory_gb": float(entry["gpu_memory_gb"]),
                        "seed": entry["seed"],
                        "hyperparams": entry["hyperparams"],
                        "dpso_ga_state": {
                            "position": entry["raw_position"],
                            "global_best_val_acc": entry["best_val_acc_so_far"]
                        }
                    }
                ]
            }
            trials_for_seed.append(trial_record)

        raw_best = self.global_best_hyperparams
        best_hparams_for_log = dict(raw_best)
        best_hparams_for_log["target_modules"] = map_target_modules_label(
            raw_best.get("target_modules", [])
        )

        seed_results = {
            "seed": self.seed,
            "trials": trials_for_seed,
            "best_trial": best_trial_no,
            "best_particle": best_trial_no,
            "best_accuracy": float(self.global_best_fitness),
            "best_hyperparams": best_hparams_for_log,
            "val_acc_mean": val_acc_mean,
            "val_acc_std": val_acc_std,
            "avg_train_time_sec": avg_time,
            "avg_gpu_memory_gb": avg_gpu_memory_gb,
            "total_train_time_sec": total_time
        }

        return seed_results


In [ ]:
# MAIN EXECUTION

def save_results_to_json(results: Dict, filename: str = "dpso_ga_results.json"):
    output = [results]  # Wrap in list
    with open(filename, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\nResults saved to: {filename}")


# main runs 4 seeds, aggregates into method-level JSON
def main():
    """Main execution function"""
    print("HYBRID DPSO-GA FOR LORA HYPERPARAMETER OPTIMIZATION")
    print()

    # Step 1: Load and prepare data (done once)
    data_dict = load_and_prepare_data()

    # Step 2: Get hyperparameter space
    search_space, decode_function = lora_hyperparameter_space()

    # Use 4 seeds
    seeds = [1, 42, 999, 1234]
    total_iterations_per_seed = 10

    all_seed_results = []
    all_seed_best_vals = []
    all_trial_vals = []
    total_time_all_seeds = 0.0
    global_best_val = -float("inf")
    global_best_seed = None
    global_best_trial_no = None
    global_best_hyperparams = None

    for seed_val in seeds:
        print(f"Running DPSO-GA for seed {seed_val}")

        # Seed-specific fitness function
        def fitness_function(hyperparams: Dict, trial_no: int) -> Tuple[float, float, float]:

            val_acc, train_time, gpu_memory_gb = train_and_evaluate_lora(
                hyperparams=hyperparams,
                data_dict=data_dict,
                seed=seed_val + trial_no,  # Different training seed per trial+seed
                trial_no=trial_no
            )
            return val_acc, train_time, gpu_memory_gb

        optimizer = HybridDPSO_GA(
            fitness_function=fitness_function,
            decode_function=decode_function,
            search_space=search_space,
            n_dimensions=6,
            total_trials=total_iterations_per_seed,
            initial_population=5,  # Start with 5 particles
            omega=0.1,
            c1=0.5,
            c2=0.5,
            ga_crossover_prob=0.8,
            ga_mutation_prob=0.1,
            seed=seed_val
        )

        seed_results = optimizer.optimize(verbose=True)
        all_seed_results.append(seed_results)

        # Per-seed stats for aggregation
        best_acc = seed_results["best_accuracy"]
        all_seed_best_vals.append(best_acc)
        all_trial_vals.extend([t["particles"][0]["val_acc"] for t in seed_results["trials"]])
        total_time_all_seeds += seed_results["total_train_time_sec"]

        if best_acc > global_best_val:
            global_best_val = best_acc
            global_best_seed = seed_val
            global_best_trial_no = seed_results["best_trial"]
            global_best_hyperparams = seed_results["best_hyperparams"]

    # Overall stats across all 4 seeds
    val_acc_mean_over_all_trials = float(np.mean(all_trial_vals))
    val_acc_std_over_all_trials = float(np.std(all_trial_vals))
    val_acc_mean_of_seed_bests = float(np.mean(all_seed_best_vals))
    val_acc_std_of_seed_bests = float(np.std(all_seed_best_vals))

    overall_stats = {
        "global_best_seed": global_best_seed,
        "global_best_trial_no": global_best_trial_no,
        "global_best_val_acc": float(global_best_val),
        "global_best_hyperparams": global_best_hyperparams,
        "val_acc_mean_over_all_trials": val_acc_mean_over_all_trials,
        "val_acc_std_over_all_trials": val_acc_std_over_all_trials,
        "val_acc_mean_of_seed_bests": val_acc_mean_of_seed_bests,
        "val_acc_std_of_seed_bests": val_acc_std_of_seed_bests,
        "total_train_time_sec_over_all_seeds": float(total_time_all_seeds)
    }

    # Method-level wrapper
    method_results = {
        "method": "DPSO_GA",
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "population_size": 5,
        "num_seeds": len(seeds),        # = 4
        "seeds": seeds,
        "total_iterations_per_seed": total_iterations_per_seed,
        "num_epochs_per_trial": NUM_EPOCHS,
        "search_space": search_space,
        "seeds_results": all_seed_results,
        "overall_stats": overall_stats
    }

    # Save JSON combined results
    save_results_to_json(method_results, filename="dpso_ga_results.json")

    print("SUMMARY (across all seeds)")
    print(f"Method: {method_results['method']}")
    print(f"Num seeds: {method_results['num_seeds']}")
    print(f"Seeds: {method_results['seeds']}")
    print(f"Global best seed: {overall_stats['global_best_seed']}")
    print(f"Global best trial: {overall_stats['global_best_trial_no']}")
    print(f"Global best accuracy: {overall_stats['global_best_val_acc']:.4f}")
    print(f"Mean of seed best accuracies: {overall_stats['val_acc_mean_of_seed_bests']:.4f} "
          f"± {overall_stats['val_acc_std_of_seed_bests']:.4f}")
    print(f"Mean accuracy over all trials: {overall_stats['val_acc_mean_over_all_trials']:.4f} "
          f"± {overall_stats['val_acc_std_over_all_trials']:.4f}")
    print(f"Total training time over all seeds: {overall_stats['total_train_time_sec_over_all_seeds']:.1f}s")


if __name__ == "__main__":
    main()

HYBRID DPSO-GA FOR LORA HYPERPARAMETER OPTIMIZATION

Loading dataset...
Train subset size: 3000
Validation size: 2000
Test size: 2000
Tokenizing datasets...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset preparation complete!

Running DPSO-GA for seed 999
HYBRID DPSO-GA OPTIMIZATION
Total iterations: 10

Initializing swarm with 5 particles...
These are initial evaluations before  10 DPSO+GA iterations.

Initial Trial 1: Evaluating initial particle 1/5...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-267804096.py:79: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
188,1.184300
376,0.603600
564,0.464000


...

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-267804096.py:79: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
188,1.123100
376,0.426000
564,0.237000


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-267804096.py:79: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
188,1.094700
376,0.414400
564,0.232700


Iteration 10 summary: best=0.8935, mean=0.8907, global_best=0.9000

OPTIMIZATION COMPLETE
Seed: 999
Total objective evaluations: 105
Best validation accuracy: 0.9000

Best hyperparameters:
  learning_rate: 0.00020222092122191526
  warmup_ratio: 0.010457726297401672
  lora_r: 16
  lora_alpha: 64
  lora_dropout: 0.02727757883222399
  target_modules: ['q_lin', 'v_lin', 'ffn.lin1', 'ffn.lin2']

Results saved to: dpso_ga_results.json
SUMMARY (across all seeds)
Method: DPSO_GA
Num seeds: 1
Seeds: [999]
Global best seed: 999
Global best trial: 100
Global best accuracy: 0.9000
Mean of seed best accuracies: 0.9000 ± 0.0000
Mean accuracy over all trials: 0.8130 ± 0.0876
Total training time over all seeds: 8461.6s
